In [1]:
# src/data/preprocessor.py
import os
import torch
import logging
import requests
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from datetime import datetime, timedelta
from typing import Dict
# from src.config import THU_DUC_BOUNDS, PROCESSED_DIR

In [2]:
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
THU_DUC_BOUNDS = {
    'north': 10.8700, 'south': 10.7300, 'east': 106.8200, 'west': 106.6800
}
print(PROJECT_DIR)
print(THU_DUC_BOUNDS)

d:\2_Workspace\4_Nam_4\1_Hoc_ky_251\1_Do_An_Chuyen_Nganh\HCM_UTL\src
{'north': 10.87, 'south': 10.73, 'east': 106.82, 'west': 106.68}


In [3]:
class TrafficDataPreprocessor:
    """
    Tiền xử lý dữ liệu giao thông từ OpenStreetMap và TomTom API
    cho dự án T-GCN phân tích giao thông TP.HCM
    """
    def __init__(self, tomtom_api_key: str =  None):
        """
        Khởi tạo preprocessor
        
        Args:
            tomtom_api_key: API key của TomTom
        """
        self.tomtom_api_key = tomtom_api_key
        self.graph = None
        self.nodes_df = pd.DataFrame()
        self.edges_df = pd.DataFrame()
        self.traffic_data = {}

    def extract_osm_data(self):
        """
        Trích xuất dữ liệu đường bộ từ OpenStreetMap
        """
        print("📍 Đang tải dữ liệu OpenStreetMap cho Thủ Đức...")
        try:
            # Tạo polygon cho khu vực Thủ Đức
            place_name = "Thủ Đức, Ho Chi Minh City, Vietnam"
            
            # Lấy graph mạng từ OSM
            self.graph = ox.graph_from_place(
                place_name,
                network_type='drive',
                simplify=True
            )
            
            print(f"✅ Đã tải {len(self.graph.nodes)} nút và {len(self.graph.edges)} cạnh")
            
            # Chuyển đổi sang DataFrame
            self._convert_graph_to_dataframes()
        except Exception as e:
            print(f"Lỗi khi tải dữ liệu OSM: {e}")
            # Fallback: sử dụng bounding box
            self._load_osm_by_bbox()
    
    def _load_osm_by_bbox(self):
        """Tải dữ liệu OSM bằng bounding box"""
        print("🔄 Đang thử với bounding box...")
        self.graph = ox.graph_from_bbox(
            north=THU_DUC_BOUNDS['north'],
            south=THU_DUC_BOUNDS['south'],
            east=THU_DUC_BOUNDS['east'],
            west=THU_DUC_BOUNDS['west'],
            network_type='drive',
            simplify=True
        )
        
        print(f"✅ Đã tải {len(self.graph.nodes)} nút và {len(self.graph.edges)} cạnh")
        self._convert_graph_to_dataframes()

    def _convert_graph_to_dataframes(self):
        """Chuyển đổi NetworkX graph sang DataFrame"""
        # Nodes Data
        nodes_data = []
        for node_id, data in self.graph.nodes(data=True):
            nodes_data.append({
                "node_id": node_id,
                "lat": data['y'],
                "lon": data['x'],
                'osmid': data.get('osmid', node_id)
            })
        
        self.nodes_df = pd.DataFrame(nodes_data)
        
        # Edges DataFrame
        edges_data = []
        for u, v, key, data in self.graph.edges(data=True, keys=True):
            print(u, v, key, data)
            edges_data.append({
                'from_node': u,
                'to_node': v,
                'edge_key': key,
                'length': data.get('length', 0),
                'highway': data.get('highway', 'unclassified'),
                'maxspeed': self._parse_maxspeed(data.get('maxspeed')),
                'lanes': self._parse_lanes(data.get('lanes')),
                'osmid': data.get('osmid', f"{u}_{v}_{key}")
            })
        self.edges_df = pd.DataFrame(edges_data)
        
        print(f"📊 Nodes DataFrame: {len(self.nodes_df)} hàng")
        print(f"📊 Edges DataFrame: {len(self.edges_df)} hàng")

    def _parse_maxspeed(self, maxspeed):
        """Parse maxspeed từ OSM data"""
        if maxspeed is None:
            return 50
        
        if isinstance(maxspeed, list):
            maxspeed = maxspeed[0]
        
        try:
            speed = int(str(maxspeed).replace("km/h", "").strip())
            return speed
        except:
            return 50
    
    def _parse_lanes(self, lanes):
        """Parse số làn đường"""
        if lanes is None:
            return 2
        
        if isinstance(lanes, list):
            lanes = lanes[0]
        
        try:
            return int(lanes)
        except: 
            return 2
    
    def get_major_intersections(self, min_degree: int = 3) -> pd.DataFrame:
        """
        Lấy các nút giao thông quan trọng (có độ >= min_degree)
        
        Args:
            min_degree: Độ tối thiểu của nút (số đường giao nhau)
        
        Returns:
            DataFrame chứa các nút giao thông quan trọng
        """
        print(f"🔍 Các nút giao thông quan trọng (độ >= {min_degree})...")
        
        # Tính degree cho mỗi node
        degrees = dict(self.graph.degree())
        
        # Lọc nodes có degree cao
        important_nodes = []
        for node_id, degree in degrees.items():
            if degree >= min_degree:
                node_data = self.graph.nodes[node_id]
                important_nodes.append({
                    'node_id': node_id,
                    'lat': node_data['y'],
                    'lon': node_data['x'],
                    'degree': degree,
                    'osmid': node_data.get('osmid', node_id)
                })
        
        important_nodes_df = pd.DataFrame(important_nodes)
        print(f"✅ Tìm thấy {len(important_nodes_df)} nút giao thông quan trọng")
        
        return important_nodes_df.sort_values('degree', ascending=False)
    
    def fetch_tomtom_flow(self, important_nodes: pd.DataFrame, num_hours: int = 24) -> Dict:
        """
        Dữ liệu giao thông từ TomTom API
        """
        print(f"🚦 Dữ liệu giao thông cho {len(important_nodes)} nút...")
        
        traffic_data = {}
        raise NotImplementedError
        
        
    
    def create_adjacency_matrix(self, important_nodes: pd.DataFrame) -> torch.Tensor:
        """
        Tạo ma trận kề cho T-GCN từ các nút quan trọng
        """
        print("🔗 Tạo ma trận kề...")
        raise NotImplementedError
    
    def create_feature_matrix(self, important_nodes: pd.DataFrame) -> torch.Tensor:
        """
        Tạo ma trận đặc trưng cho T-GCN từ dữ liệu giao thông
        """
        raise NotImplementedError
    
    def visualize_network(self, important_nodes: pd.DataFrame, save_path: str = None):
        """
        Visualize mạng giao thông
        """
        raise NotImplementedError
    
    def save_processed_data(self, important_nodes: pd.DataFrame,
                            adj_matrix: torch.Tensor,
                            features: torch.Tensor,
                            output_dir: str = "processed"):
        """
        Lưu dữ liệu đã xử lý
        """
        raise NotImplementedError
    
        

In [ ]:
pre = TrafficDataPreprocessor()
pre.extract_osm_data()
print("Graph info:", pre.graph)
print("Số nút:", len(pre.graph.nodes))
print("Số cạnh:", len(pre.graph.edges))
print("DataFrame nodes:", pre.nodes_df.head())
print("DataFrame edges:", pre.edges_df.head())

📍 Đang tải dữ liệu OpenStreetMap cho Thủ Đức...
✅ Đã tải 1005 nút và 2270 cạnh
366371768 366401628 0 {'osmid': 32582492, 'highway': 'residential', 'name': 'Chu Mạnh Trinh', 'oneway': False, 'reversed': False, 'length': np.float64(111.89327583823923)}
366371768 366458086 0 {'osmid': 608169579, 'highway': 'residential', 'lanes': '2', 'maxspeed': '50', 'name': 'Bác Ái', 'oneway': False, 'reversed': True, 'length': np.float64(143.628566688166)}
366371768 366477012 0 {'osmid': 1221451185, 'highway': 'residential', 'name': 'Chu Mạnh Trinh', 'oneway': False, 'reversed': True, 'length': np.float64(114.74051852310515)}
366371768 11813524962 0 {'osmid': 1221451184, 'highway': 'residential', 'lanes': '2', 'maxspeed': '50', 'name': 'Bác Ái', 'oneway': False, 'reversed': False, 'length': np.float64(207.96192086310538), 'geometry': <LINESTRING (106.767 10.848, 106.768 10.849, 106.768 10.85)>}
366373335 8814518226 0 {'osmid': 952323558, 'highway': 'residential', 'oneway': False, 'reversed': False, 'l